# PiRD для горизонтальной двухфазной фильтрации

Этот блокнот переносит идею **PiRD (Physics-informed Residual Diffusion)** на задачу из вашего блокнота PhySR.  
Численный генератор high-resolution решений и все физические члены для сравнения берутся из уже реализованного решателя по уравнениям (1.2.1)–(1.2.11).

Что делает блокнот:

- генерирует HR-данные тем же решателем, что и в исходном ноутбуке;
- строит LR-наблюдения через downsampling/corruption в стиле PiRD;
- обучает **conditional residual diffusion** модель на восстановление HR-поля из LR;
- использует те же физические ограничения и метрики для сравнения с baseline-upsample и с PhySR-подходом.

> Ниже архитектура и pipeline адаптированы под задачу фильтрации из исходного ноутбука и повторяют основные идеи репозитория PiRD: residual diffusion, conditional reconstruction, corruption modes `skip / average / portion`, physics-informed loss.


## 1. Импорты и базовая конфигурация

In [ ]:
import math, time, copy, os, random, json, inspect, platform, tracemalloc
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace

import h5py
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import psutil
except ImportError:
    psutil = None


In [ ]:

@dataclass
class Config:
    nx_hr: int = 64
    ny_hr: int = 64
    nt_hr: int = 14
    s_up: int = 4
    t_up: int = 2

    ctx_frames: int = 3
    diffusion_steps: int = 20
    beta_start: float = 1e-4
    beta_end: float = 2e-2
    predict_type: str = "eps"
    residual_clip_p: float = 0.08
    residual_clip_s: float = 0.12

    corruption_method: str = "skip"
    corruption_scale: int = 4
    corruption_portion: float = 0.05

    mu_w: float = 1.0
    mu_n: float = 4.0
    Swc: float = 0.20
    Smax: float = 0.90
    n_w: float = 3.0
    n_n: float = 2.0

    Pinj: float = 1.00
    Pprod: float = 0.22
    Pgamma: float = 0.22
    S_plus: float = 0.23

    H0: float = 1.0
    m0: float = 0.24

    dt: float = 0.0025
    sat_substeps: int = 3
    n_pressure_iters: int = 260
    well_radius: int = 2
    n_cases: int = 20
    cfl: float = 0.45
    epsilon_sat: float = 1e-4
    epsilon_front: float = 0.05
    well_radius_phys: float = 0.02

    batch_size: int = 1
    lr: float = 1e-4
    weight_decay: float = 1e-6
    epochs: int = 40
    physics_start_epoch: int = 16
    physics_ramp_epochs: int = 10
    max_phys_w: float = 2e-4
    loss_recon: float = 1.0
    loss_p_grad: float = 0.10
    loss_s_grad: float = 0.20
    loss_s_front: float = 0.30
    loss_residual_reg: float = 0.01
    loss_identity: float = 0.01
    loss_diffusion: float = 1.0

    early_stopping_patience: int = 8
    early_stopping_min_delta: float = 5e-5
    lr_scheduler_factor: float = 0.5
    lr_scheduler_patience: int = 3
    lr_scheduler_min: float = 1e-5

    front_theta: float = 0.55
    front_k: float = 28.0

    target_ff_P: float = 5.0
    target_ff_S: float = 8.0
    target_ff_joint: float = 7.0

cfg = Config()
cfg.nx_lr = cfg.nx_hr // cfg.s_up
cfg.ny_lr = cfg.ny_hr // cfg.s_up
cfg.nt_lr = math.ceil(cfg.nt_hr / cfg.t_up)
cfg.dx = 1.0 / (cfg.nx_hr - 1)
cfg.dy = 1.0 / (cfg.ny_hr - 1)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
print(asdict(cfg))


## 2. Численный HR-решатель из исходного блокнота

Ниже блоки без изменения переносятся из вашего ноутбука и используются как:
- генератор синтетических HR-решений;
- численный reference-solver для физической регуляризации и сравнения.


In [ ]:
def smooth_noise(ny, nx, seed, sweeps=22):
    rng = np.random.default_rng(seed)
    z = rng.normal(size=(ny, nx))
    for _ in range(sweeps):
        z = (
            4.0 * z +
            np.roll(z, 1, 0) + np.roll(z, -1, 0) +
            np.roll(z, 1, 1) + np.roll(z, -1, 1)
        ) / 8.0
    z = (z - z.mean()) / (z.std() + 1e-8)
    return z


def circle_mask(ny, nx, cx, cy, radius):
    yy, xx = np.mgrid[0:ny, 0:nx]
    return (xx - cx) ** 2 + (yy - cy) ** 2 <= radius ** 2


def relperm_np(S, cfg):
    Se = np.clip((S - cfg.Swc) / (cfg.Smax - cfg.Swc), 0.0, 1.0)
    Kw = np.where(S <= cfg.Swc, 0.0, Se ** cfg.n_w)
    Kn = np.where(S >= cfg.Smax, 0.0, (1.0 - Se) ** cfg.n_n)
    return Kw.astype(np.float32), Kn.astype(np.float32)


def fw_from_s_np(S, cfg):
    Kw, Kn = relperm_np(S, cfg)
    lam_w = Kw / cfg.mu_w
    lam_n = Kn / cfg.mu_n
    lam_t = lam_w + lam_n + 1e-10
    return (lam_w / lam_t).astype(np.float32)


def make_coefficients(seed, cfg):
    x = np.linspace(0.0, 1.0, cfg.nx_hr)
    y = np.linspace(0.0, 1.0, cfg.ny_hr)
    X, Y = np.meshgrid(x, y)

    Kf = smooth_noise(cfg.ny_hr, cfg.nx_hr, seed=100 + seed, sweeps=24)
    Hf = smooth_noise(cfg.ny_hr, cfg.nx_hr, seed=200 + seed, sweeps=16)
    mf = smooth_noise(cfg.ny_hr, cfg.nx_hr, seed=300 + seed, sweeps=16)

    K = np.exp(0.45 * Kf)
    K /= K.mean()

    H = cfg.H0 * (1.0 + 0.12 * Hf)
    H = np.clip(H, 0.75, 1.25)

    m = cfg.m0 * (1.0 + 0.08 * mf)
    m = np.clip(m, 0.18, 0.30)

    inj_cx, inj_cy = int(0.18 * cfg.nx_hr), int(0.52 * cfg.ny_hr)
    prod_cx, prod_cy = int(0.82 * cfg.nx_hr), int(0.50 * cfg.ny_hr)

    inj_mask = circle_mask(cfg.ny_hr, cfg.nx_hr, inj_cx, inj_cy, cfg.well_radius)
    prod_mask = circle_mask(cfg.ny_hr, cfg.nx_hr, prod_cx, prod_cy, cfg.well_radius)

    S0 = cfg.Swc + 0.012
    S0 = S0 + 0.11 * np.exp(-((X - 0.18)**2 / 0.022 + (Y - 0.52)**2 / 0.034))
    S0 = S0 + 0.02 * np.exp(-((X - 0.30)**2 / 0.040 + (Y - 0.65)**2 / 0.045))
    S0 = np.clip(S0, cfg.Swc + 1e-4, cfg.Smax - 0.10)
    S0[inj_mask] = cfg.Smax

    return K.astype(np.float32), H.astype(np.float32), m.astype(np.float32), inj_mask, prod_mask, S0.astype(np.float32)


In [ ]:

def _well_centers_from_masks(inj_mask, prod_mask):
    centers = []
    for mask, kind in [(inj_mask, "inj"), (prod_mask, "prod")]:
        pts = np.argwhere(mask)
        if len(pts) == 0:
            continue
        cy, cx = pts.mean(axis=0)
        centers.append((float(cx), float(cy), kind))
    return centers


def _zeta_from_r2(r2, cfg):
    if r2 <= 0.25:
        return 0.5 * np.pi / max(np.log(max(cfg.dx, cfg.dy) / max(cfg.well_radius_phys, 1e-6)), 1e-6)
    if abs(r2 - 1.25) < 1e-12:
        return 1.3378
    if abs(r2 - 2.25) < 1e-12:
        return 0.9284
    if r2 > 3.0:
        return 1.0
    # smooth fallback between tabulated values
    if r2 < 1.25:
        t = (r2 - 0.25) / 1.0
        return (1.0 - t) * (0.5 * np.pi / max(np.log(max(cfg.dx, cfg.dy) / max(cfg.well_radius_phys, 1e-6)), 1e-6)) + t * 1.3378
    t = (r2 - 1.25) / 1.0
    return (1.0 - t) * 1.3378 + t * 0.9284


def _zeta_face_array(shape, orientation, inj_mask, prod_mask, cfg):
    ny, nx = shape
    wells = _well_centers_from_masks(inj_mask, prod_mask)
    if orientation == "x":
        zeta = np.ones((ny, nx + 1), dtype=np.float32)
        for y in range(ny):
            for f in range(nx + 1):
                fx, fy = f - 0.5, y
                best = 1.0
                for cx, cy, _ in wells:
                    r2 = (cx - fx) ** 2 + (cy - fy) ** 2
                    best = _zeta_from_r2(r2, cfg) if r2 <= 3.0 else best
                    if r2 <= 3.0:
                        break
                zeta[y, f] = best
        return zeta
    zeta = np.ones((ny + 1, nx), dtype=np.float32)
    for f in range(ny + 1):
        for x in range(nx):
            fx, fy = x, f - 0.5
            best = 1.0
            for cx, cy, _ in wells:
                r2 = (cx - fx) ** 2 + (cy - fy) ** 2
                best = _zeta_from_r2(r2, cfg) if r2 <= 3.0 else best
                if r2 <= 3.0:
                    break
            zeta[f, x] = best
    return zeta


def solve_pressure(S, K, H, inj_mask, prod_mask, cfg):
    ny, nx = S.shape
    P = np.full((ny, nx), cfg.Pgamma, dtype=np.float32)
    P[inj_mask] = cfg.Pinj
    P[prod_mask] = cfg.Pprod

    Kw, Kn = relperm_np(np.clip(S, cfg.Swc, cfg.Smax), cfg)
    lam_t = Kw / cfg.mu_w + Kn / cfg.mu_n
    a = H * K * lam_t + 1e-10

    zeta_x = _zeta_face_array(S.shape, "x", inj_mask, prod_mask, cfg)
    zeta_y = _zeta_face_array(S.shape, "y", inj_mask, prod_mask, cfg)

    fixed = np.zeros((ny, nx), dtype=bool)
    fixed[:, 0] = fixed[:, -1] = True
    fixed[0, :] = fixed[-1, :] = True
    fixed[inj_mask] = True
    fixed[prod_mask] = True

    omega = 0.8
    for _ in range(cfg.n_pressure_iters):
        Pn = P.copy()
        ae = zeta_x[1:-1, 2:nx] * 0.5 * (a[1:-1, 1:-1] + a[1:-1, 2:])
        aw = zeta_x[1:-1, 1:nx-1] * 0.5 * (a[1:-1, 1:-1] + a[1:-1, :-2])
        an = zeta_y[2:ny, 1:-1] * 0.5 * (a[1:-1, 1:-1] + a[2:, 1:-1])
        ass = zeta_y[1:ny-1, 1:-1] * 0.5 * (a[1:-1, 1:-1] + a[:-2, 1:-1])
        denom = ae + aw + an + ass + 1e-12
        core = (ae * P[1:-1, 2:] + aw * P[1:-1, :-2] + an * P[2:, 1:-1] + ass * P[:-2, 1:-1]) / denom
        Pn[1:-1, 1:-1] = (1.0 - omega) * P[1:-1, 1:-1] + omega * core

        Pn[:, 0] = cfg.Pgamma
        Pn[:, -1] = cfg.Pgamma
        Pn[0, :] = cfg.Pgamma
        Pn[-1, :] = cfg.Pgamma
        Pn[inj_mask] = cfg.Pinj
        Pn[prod_mask] = cfg.Pprod
        P = Pn

    return P.astype(np.float32), a.astype(np.float32), fixed


def total_flux_faces_np(P, a, cfg, inj_mask=None, prod_mask=None):
    ny, nx = P.shape
    if inj_mask is None:
        inj_mask = np.zeros_like(P, dtype=bool)
    if prod_mask is None:
        prod_mask = np.zeros_like(P, dtype=bool)

    zeta_x = _zeta_face_array(P.shape, "x", inj_mask, prod_mask, cfg)
    zeta_y = _zeta_face_array(P.shape, "y", inj_mask, prod_mask, cfg)

    qx = np.zeros((ny, nx + 1), dtype=np.float32)
    qy = np.zeros((ny + 1, nx), dtype=np.float32)

    a_x = 0.5 * (a[:, :-1] + a[:, 1:])
    qx[:, 1:nx] = -zeta_x[:, 1:nx] * a_x * (P[:, 1:] - P[:, :-1]) / cfg.dx
    qx[:, 0] = -zeta_x[:, 0] * a[:, 0] * (P[:, 0] - cfg.Pgamma) / (0.5 * cfg.dx)
    qx[:, nx] = -zeta_x[:, nx] * a[:, -1] * (cfg.Pgamma - P[:, -1]) / (0.5 * cfg.dx)

    a_y = 0.5 * (a[:-1, :] + a[1:, :])
    qy[1:ny, :] = -zeta_y[1:ny, :] * a_y * (P[1:, :] - P[:-1, :]) / cfg.dy
    qy[0, :] = -zeta_y[0, :] * a[0, :] * (P[0, :] - cfg.Pgamma) / (0.5 * cfg.dy)
    qy[ny, :] = -zeta_y[ny, :] * a[-1, :] * (cfg.Pgamma - P[-1, :]) / (0.5 * cfg.dy)
    return qx, qy


def _fractional_linear_face_value(j_prev, j_curr, j_opp, cfg):
    eps = cfg.epsilon_sat
    front = cfg.epsilon_front
    if j_curr >= cfg.Smax - front:
        return cfg.Smax
    if j_curr <= cfg.Swc + front:
        return cfg.Swc
    if j_prev >= j_curr:
        F = 0.5 * (j_prev + j_curr) * j_curr / max(j_prev, eps)
    else:
        F = 0.5 * (1.0 + j_curr - (1.0 - j_curr) ** 2 / max(1.0 - j_prev, eps))
    lo = min(j_curr, j_opp)
    hi = max(j_curr, j_opp)
    if F < lo or F > hi:
        return j_curr
    return float(np.clip(F, cfg.Swc, cfg.Smax))


def _face_saturation_x(S, qx, cfg):
    ny, nx = S.shape
    Sf = np.zeros((ny, nx + 1), dtype=np.float32)

    for y in range(ny):
        if qx[y, 0] > 0.0:
            Sf[y, 0] = cfg.S_plus
        else:
            Sf[y, 0] = S[y, 0]

        for f in range(1, nx):
            if qx[y, f] >= 0.0:
                j_curr = S[y, f - 1]
                j_prev = S[y, max(f - 2, 0)]
                j_opp = S[y, f]
            else:
                j_curr = S[y, f]
                j_prev = S[y, min(f + 1, nx - 1)]
                j_opp = S[y, f - 1]
            Sf[y, f] = _fractional_linear_face_value(j_prev, j_curr, j_opp, cfg)

        if qx[y, nx] < 0.0:
            Sf[y, nx] = cfg.S_plus
        else:
            Sf[y, nx] = S[y, -1]

    return np.clip(Sf, cfg.Swc, cfg.Smax)


def _face_saturation_y(S, qy, cfg):
    ny, nx = S.shape
    Sf = np.zeros((ny + 1, nx), dtype=np.float32)

    for x in range(nx):
        if qy[0, x] > 0.0:
            Sf[0, x] = cfg.S_plus
        else:
            Sf[0, x] = S[0, x]

        for f in range(1, ny):
            if qy[f, x] >= 0.0:
                j_curr = S[f - 1, x]
                j_prev = S[max(f - 2, 0), x]
                j_opp = S[f, x]
            else:
                j_curr = S[f, x]
                j_prev = S[min(f + 1, ny - 1), x]
                j_opp = S[f - 1, x]
            Sf[f, x] = _fractional_linear_face_value(j_prev, j_curr, j_opp, cfg)

        if qy[ny, x] < 0.0:
            Sf[ny, x] = cfg.S_plus
        else:
            Sf[ny, x] = S[-1, x]

    return np.clip(Sf, cfg.Swc, cfg.Smax)


def _apply_sector_well_update(S_new, S_old, qx, qy, cfg, prod_mask, H):
    pts = np.argwhere(prod_mask)
    if len(pts) == 0:
        return S_new
    cy, cx = np.round(pts.mean(axis=0)).astype(int)
    Hm = float(np.mean(H[prod_mask])) if np.any(prod_mask) else float(H[cy, cx])

    axial = [(1, 0), (-1, 0), (0, 1), (0, -1)]
    for dxs, dys in axial:
        x = cx + dxs
        y = cy + dys
        if not (0 <= x < S_new.shape[1] and 0 <= y < S_new.shape[0]):
            continue
        if prod_mask[y, x]:
            continue
        if dxs == 1:
            v_face = qx[y, min(x + 1, qx.shape[1] - 1)]
            s_face = _face_saturation_x(S_old, qx, cfg)[y, min(x + 1, qx.shape[1] - 1)]
        elif dxs == -1:
            v_face = qx[y, x]
            s_face = _face_saturation_x(S_old, qx, cfg)[y, x]
        elif dys == 1:
            v_face = qy[min(y + 1, qy.shape[0] - 1), x]
            s_face = _face_saturation_y(S_old, qy, cfg)[min(y + 1, qy.shape[0] - 1), x]
        else:
            v_face = qy[y, x]
            s_face = _face_saturation_y(S_old, qy, cfg)[y, x]
        f_cell = fw_from_s_np(np.array([[S_old[y, x]]], dtype=np.float32), cfg)[0, 0]
        f_face = fw_from_s_np(np.array([[s_face]], dtype=np.float32), cfg)[0, 0]
        S_new[y, x] = np.clip(S_old[y, x] + 4.0 * cfg.dt * v_face * (f_face - f_cell) / (3.0 * Hm * max(cfg.dx, cfg.dy) + 1e-12), cfg.Swc, cfg.Smax)

    diags = [(1, 1), (1, -1), (-1, 1), (-1, -1)]
    Sfx = _face_saturation_x(S_old, qx, cfg)
    Sfy = _face_saturation_y(S_old, qy, cfg)
    for dxs, dys in diags:
        x = cx + dxs
        y = cy + dys
        if not (0 <= x < S_new.shape[1] and 0 <= y < S_new.shape[0]):
            continue
        if prod_mask[y, x]:
            continue
        v1 = qx[y, x + 1] if dxs > 0 else qx[y, x]
        s1 = Sfx[y, x + 1] if dxs > 0 else Sfx[y, x]
        v2 = qy[y + 1, x] if dys > 0 else qy[y, x]
        s2 = Sfy[y + 1, x] if dys > 0 else Sfy[y, x]
        f_cell = fw_from_s_np(np.array([[S_old[y, x]]], dtype=np.float32), cfg)[0, 0]
        f1 = fw_from_s_np(np.array([[s1]], dtype=np.float32), cfg)[0, 0]
        f2 = fw_from_s_np(np.array([[s2]], dtype=np.float32), cfg)[0, 0]
        incr = 2.0 * cfg.dt * (v1 * (f1 - f_cell) + v2 * (f2 - f_cell)) / (3.0 * Hm * max(cfg.dx, cfg.dy) + 1e-12)
        S_new[y, x] = np.clip(S_old[y, x] + incr, cfg.Swc, cfg.Smax)

    return S_new


def water_flux_faces_tvd_np(S, qx, qy, cfg):
    Sx = _face_saturation_x(S, qx, cfg)
    Sy = _face_saturation_y(S, qy, cfg)
    Fwx = fw_from_s_np(Sx, cfg) * qx
    Fwy = fw_from_s_np(Sy, cfg) * qy
    return Fwx.astype(np.float32), Fwy.astype(np.float32)


def sat_rhs_tvd_np(S, P, K, H, m, inj_mask, prod_mask, cfg):
    S_work = np.clip(S, cfg.Swc, cfg.Smax).astype(np.float32)
    S_work[inj_mask] = cfg.Smax

    Kw, Kn = relperm_np(S_work, cfg)
    lam_t = Kw / cfg.mu_w + Kn / cfg.mu_n
    a = H * K * lam_t + 1e-10
    qx, qy = total_flux_faces_np(P, a, cfg, inj_mask=inj_mask, prod_mask=prod_mask)
    Fwx, Fwy = water_flux_faces_tvd_np(S_work, qx, qy, cfg)
    div_w = (Fwx[:, 1:] - Fwx[:, :-1]) / cfg.dx + (Fwy[1:, :] - Fwy[:-1, :]) / cfg.dy

    rhs = -div_w / (m * H + 1e-10)
    rhs[inj_mask] = 0.0
    rhs[prod_mask] = 0.0
    return rhs.astype(np.float32), qx, qy, Fwx, Fwy


def update_saturation(S, P, K, H, m, inj_mask, prod_mask, cfg):
    Kw, Kn = relperm_np(np.clip(S, cfg.Swc, cfg.Smax), cfg)
    lam_t = Kw / cfg.mu_w + Kn / cfg.mu_n
    a = H * K * lam_t + 1e-10
    qx, qy = total_flux_faces_np(P, a, cfg, inj_mask=inj_mask, prod_mask=prod_mask)
    vel = max(np.max(np.abs(qx)), np.max(np.abs(qy))) / (np.min(m * H) + 1e-10)
    dt = min(cfg.dt, cfg.cfl * min(cfg.dx, cfg.dy) / (vel + 1e-10))

    rhs, qx, qy, _, _ = sat_rhs_tvd_np(S, P, K, H, m, inj_mask, prod_mask, cfg)
    S_new = np.clip(S + dt * rhs, cfg.Swc, cfg.Smax).astype(np.float32)
    S_new[inj_mask] = cfg.Smax
    S_new = _apply_sector_well_update(S_new, np.clip(S, cfg.Swc, cfg.Smax), qx, qy, cfg, prod_mask, H)
    S_new[inj_mask] = cfg.Smax
    return np.clip(S_new, cfg.Swc, cfg.Smax).astype(np.float32)


def compute_producer_rates(P, S, K, H, prod_mask, cfg):
    Kw, Kn = relperm_np(np.clip(S, cfg.Swc, cfg.Smax), cfg)
    lam_t = Kw / cfg.mu_w + Kn / cfg.mu_n
    a = H * K * lam_t + 1e-10
    qx, qy = total_flux_faces_np(P, a, cfg, prod_mask=prod_mask)
    Fwx, Fwy = water_flux_faces_tvd_np(np.clip(S, cfg.Swc, cfg.Smax), qx, qy, cfg)

    q_total = 0.0
    q_water = 0.0
    ny, nx = P.shape
    for y, x in np.argwhere(prod_mask):
        if x > 0 and not prod_mask[y, x - 1]:
            q_total += qx[y, x]
            q_water += Fwx[y, x]
        if x < nx - 1 and not prod_mask[y, x + 1]:
            q_total -= qx[y, x + 1]
            q_water -= Fwx[y, x + 1]
        if y > 0 and not prod_mask[y - 1, x]:
            q_total += qy[y, x]
            q_water += Fwy[y, x]
        if y < ny - 1 and not prod_mask[y + 1, x]:
            q_total -= qy[y + 1, x]
            q_water -= Fwy[y + 1, x]
    return float(q_total), float(q_water)


def simulate_case(seed, cfg):
    K, H, m, inj_mask, prod_mask, S = make_coefficients(seed, cfg)
    P_seq, S_seq, qc_seq, qw_seq = [], [], [], []

    for _ in range(cfg.nt_hr):
        P, a, fixed = solve_pressure(S, K, H, inj_mask, prod_mask, cfg)
        qc, qw = compute_producer_rates(P, S, K, H, prod_mask, cfg)
        P_seq.append(P.copy())
        S_seq.append(S.copy())
        qc_seq.append(qc)
        qw_seq.append(qw)

        for _ in range(cfg.sat_substeps):
            S = update_saturation(S, P, K, H, m, inj_mask, prod_mask, cfg)

    hr = np.stack([np.stack(P_seq, axis=0), np.stack(S_seq, axis=0)], axis=1)
    meta = {
        "K": K, "H": H, "m": m,
        "inj": inj_mask, "prod": prod_mask,
        "Qc": np.array(qc_seq, dtype=np.float32),
        "Qw": np.array(qw_seq, dtype=np.float32),
        "watercut": np.array(qw_seq, dtype=np.float32) / (np.array(qc_seq, dtype=np.float32) + 1e-10),
    }
    return hr.astype(np.float32), meta


In [ ]:
# ============================================================
# HDF5 dataset configuration, aligned with the reference notebook
# ============================================================
DATA_DIR = Path(os.environ.get(
    "FILTRATION_DATA_DIR",
    "/kaggle/input/datasets/sofiagamershmidt/synthetic-filtration-x8-resolution",
))
NAME = os.environ.get("FILTRATION_DATA_NAME", "synth128_x8_t3")

CONFIG_PATH = DATA_DIR / "config.json"
TRAIN_H5 = DATA_DIR / f"{NAME}_train.h5"
VAL_H5 = DATA_DIR / f"{NAME}_val.h5"
TEST_H5 = DATA_DIR / f"{NAME}_test.h5"


def make_cfg_from_json(config_path, base_cls=Config):
    with open(config_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    sig = inspect.signature(base_cls)
    allowed = set(sig.parameters.keys())
    init_kwargs = {k: v for k, v in data.items() if k in allowed}
    cfg = base_cls(**init_kwargs)

    for k, v in data.items():
        if not hasattr(cfg, k):
            setattr(cfg, k, v)

    return cfg


def sync_cfg_with_h5(cfg, h5_path):
    with h5py.File(h5_path, "r") as f:
        _, nt_hr, _, ny_hr, nx_hr = f["hr"].shape
        _, nt_lr, _, ny_lr, nx_lr = f["lr"].shape

    cfg.nx_hr = int(nx_hr)
    cfg.ny_hr = int(ny_hr)
    cfg.nt_hr = int(nt_hr)
    cfg.nx_lr = int(nx_lr)
    cfg.ny_lr = int(ny_lr)
    cfg.nt_lr = int(nt_lr)
    cfg.s_up = max(1, int(round(nx_hr / nx_lr)))
    cfg.t_up = max(1, int(round((nt_hr - 1) / max(nt_lr - 1, 1))))
    cfg.ctx_frames = min(int(getattr(cfg, "ctx_frames", 3)), cfg.nt_hr)
    cfg.dx = 1.0 / max(1, cfg.nx_hr - 1)
    cfg.dy = 1.0 / max(1, cfg.ny_hr - 1)
    cfg.device = "cuda" if torch.cuda.is_available() else "cpu"
    return cfg


def compute_mean_std_h5(path, chunk_size=16):
    with h5py.File(path, "r") as f:
        hr = f["hr"]
        n, t, c, h, w = hr.shape
        total = n * t * h * w
        sums = np.zeros(c, dtype=np.float64)
        sqs = np.zeros(c, dtype=np.float64)

        for i0 in range(0, n, chunk_size):
            x = hr[i0:i0 + chunk_size].astype(np.float64)
            sums += x.sum(axis=(0, 1, 3, 4))
            sqs += (x ** 2).sum(axis=(0, 1, 3, 4))

        mean_np = sums / total
        var_np = sqs / total - mean_np ** 2
        std_np = np.sqrt(np.maximum(var_np, 1e-12)) + 1e-6

    return torch.tensor(mean_np, dtype=torch.float32), torch.tensor(std_np, dtype=torch.float32)


def load_or_compute_normalization(data_dir, train_h5):
    norm_path = Path(data_dir) / "normalization.json"

    if norm_path.exists():
        with open(norm_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return torch.tensor(data["mean"], dtype=torch.float32), torch.tensor(data["std"], dtype=torch.float32)

    return compute_mean_std_h5(train_h5)


def lr_sequence_to_condition(lr_state, cfg):
    x = torch.as_tensor(lr_state, dtype=torch.float32)
    single = x.ndim == 4
    if single:
        x = x.unsqueeze(0)

    b, t_lr, ch, ny_lr, nx_lr = x.shape
    x = x.reshape(b * t_lr * ch, 1, ny_lr, nx_lr)
    x = F.interpolate(x, size=(cfg.ny_hr, cfg.nx_hr), mode="bicubic", align_corners=True)
    x = x.reshape(b, t_lr, ch, cfg.ny_hr, cfg.nx_hr)

    if t_lr != cfg.nt_hr:
        x = x.permute(0, 2, 3, 4, 1).reshape(b * ch * cfg.ny_hr * cfg.nx_hr, 1, t_lr)
        if t_lr > 1:
            x = F.interpolate(x, size=cfg.nt_hr, mode="linear", align_corners=True)
        else:
            x = x.repeat_interleave(cfg.nt_hr, dim=-1)
        x = x.reshape(b, ch, cfg.ny_hr, cfg.nx_hr, cfg.nt_hr).permute(0, 4, 1, 2, 3)

    x[:, :, 1] = x[:, :, 1].clamp(cfg.Swc + 1e-4, cfg.Smax)
    return x[0] if single else x


required_paths = [CONFIG_PATH, TRAIN_H5, VAL_H5, TEST_H5]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Dataset files are missing. Set FILTRATION_DATA_DIR / FILTRATION_DATA_NAME or update DATA_DIR and NAME.\n"
        + "\n".join(missing)
    )

cfg = make_cfg_from_json(CONFIG_PATH)
cfg = sync_cfg_with_h5(cfg, TRAIN_H5)
device = cfg.device
mean, std = load_or_compute_normalization(DATA_DIR, TRAIN_H5)

print("Dataset:", NAME)
print("DATA_DIR:", DATA_DIR)
print("HR:", (cfg.nt_hr, 2, cfg.ny_hr, cfg.nx_hr))
print("LR:", (cfg.nt_lr, 2, cfg.ny_lr, cfg.nx_lr))
print("s_up:", cfg.s_up, "t_up:", cfg.t_up, "ctx_frames:", cfg.ctx_frames)
print("device:", device)
print("mean:", mean)
print("std :", std)


In [ ]:
def make_case(seed, cfg):
    state, meta = simulate_case(seed, cfg)
    return {"state": state, **meta}

## 3. Генерация одного примера и sanity-check

In [ ]:
with h5py.File(TRAIN_H5, "r") as f:
    hr_case = torch.as_tensor(f["hr"][0], dtype=torch.float32)
    lr_case = torch.as_tensor(f["lr"][0], dtype=torch.float32)
    K_case = f["K"][0]
    H_case = f["H"][0]
    m_case = f["m"][0]

cond_case = lr_sequence_to_condition(lr_case, cfg)

print("hr state shape:", tuple(hr_case.shape))
print("lr state shape:", tuple(lr_case.shape))
print("condition shape:", tuple(cond_case.shape))
print("K/H/m shapes:", K_case.shape, H_case.shape, m_case.shape)

fig, ax = plt.subplots(2, 3, figsize=(12, 7))
t_show = hr_case.shape[0] - 1

ax[0, 0].imshow(hr_case[0, 0], cmap="viridis"); ax[0, 0].set_title("P HR t0"); ax[0, 0].axis("off")
ax[0, 1].imshow(hr_case[t_show, 0], cmap="viridis"); ax[0, 1].set_title("P HR tT"); ax[0, 1].axis("off")
ax[0, 2].imshow(cond_case[t_show, 0], cmap="viridis"); ax[0, 2].set_title("P LR→cond tT"); ax[0, 2].axis("off")
ax[1, 0].imshow(hr_case[0, 1], cmap="turbo"); ax[1, 0].set_title("S HR t0"); ax[1, 0].axis("off")
ax[1, 1].imshow(hr_case[t_show, 1], cmap="turbo"); ax[1, 1].set_title("S HR tT"); ax[1, 1].axis("off")
ax[1, 2].imshow(cond_case[t_show, 1], cmap="turbo"); ax[1, 2].set_title("S LR→cond tT"); ax[1, 2].axis("off")
plt.tight_layout()


## 4. Corruption/upscaling в стиле PiRD

In [ ]:

def nearest_fill_zeros(arr_2d):
    arr = arr_2d.copy()
    zero_idx = np.argwhere(arr == 0)
    nz_idx = np.argwhere(arr != 0)
    if len(zero_idx) == 0 or len(nz_idx) == 0:
        return arr
    for zi in zero_idx:
        d2 = ((nz_idx - zi[None, :]) ** 2).sum(axis=1)
        arr[tuple(zi)] = arr[tuple(nz_idx[np.argmin(d2)])]
    return arr

def corrupt_one_frame(frame_hr, cfg):
    x = torch.tensor(frame_hr).float().unsqueeze(0)
    method = cfg.corruption_method
    scale = cfg.corruption_scale

    if method == "skip":
        y = x[:, :, ::scale, ::scale]
        up = F.interpolate(y, size=(cfg.ny_hr, cfg.nx_hr), mode="bicubic", align_corners=True)
    elif method == "average":
        y = F.avg_pool2d(x, kernel_size=scale, stride=scale, padding=0)
        up = F.interpolate(y, size=(cfg.ny_hr, cfg.nx_hr), mode="bicubic", align_corners=True)
    elif method == "portion":
        N, C, H, W = x.shape
        total = H * W
        keep = int(total * cfg.corruption_portion)
        flat_idx = torch.randperm(total)[:keep]
        mask = torch.zeros((N, C, total), dtype=torch.bool)
        mask[:, :, flat_idx] = True
        mask = mask.view(N, C, H, W)
        sparse = x * mask.float()
        sparse_np = sparse[0].numpy()
        filled = np.stack([nearest_fill_zeros(sparse_np[c]) for c in range(C)], axis=0)
        up = torch.tensor(filled).float().unsqueeze(0)
    else:
        raise ValueError(method)

    return up[0]

def corrupt_sequence_hr_to_condition(hr_state, cfg):
    return torch.stack([corrupt_one_frame(hr_state[t], cfg) for t in range(hr_state.shape[0])], dim=0)


## 5. Dataset для PiRD

Здесь окно из `ctx_frames=3` подряд идущих кадров используется как в репозитории PiRD для three-channel режима.  
Модель обучается восстанавливать **HR residual** относительно bicubic/nearest baseline.


In [ ]:
class H5FiltrationPiRDDataset(Dataset):
    def __init__(self, path, cfg):
        self.path = str(path)
        self.cfg = cfg
        self.window = int(cfg.ctx_frames)

        with h5py.File(self.path, "r") as f:
            self.length = int(f["hr"].shape[0])
            self.nt_hr = int(f["hr"].shape[1])
            self.available_keys = set(f.keys())

        if self.window > self.nt_hr:
            raise ValueError(f"ctx_frames={self.window} is larger than nt_hr={self.nt_hr}")

        self.items = [
            (case_idx, t0)
            for case_idx in range(self.length)
            for t0 in range(self.nt_hr - self.window + 1)
        ]

    def __len__(self):
        return len(self.items)

    def _to_tensor(self, key, arr):
        if key in {"inj", "prod", "valid_mask", "valid_lr_mask", "breakthrough_reached"}:
            return torch.as_tensor(arr, dtype=torch.bool)
        if key in {"breakthrough_frame", "n_valid_frames"}:
            return torch.as_tensor(arr, dtype=torch.long)
        return torch.as_tensor(arr, dtype=torch.float32)

    def _slice_optional_time(self, arr, t0):
        if hasattr(arr, "shape") and len(arr.shape) > 0 and arr.shape[0] == self.nt_hr:
            return arr[t0:t0 + self.window]
        return arr

    def __getitem__(self, idx):
        case_idx, t0 = self.items[idx]
        w = self.window

        with h5py.File(self.path, "r") as f:
            hr_full = torch.as_tensor(f["hr"][case_idx], dtype=torch.float32)
            lr_full = torch.as_tensor(f["lr"][case_idx], dtype=torch.float32)
            cond_full = lr_sequence_to_condition(lr_full, self.cfg)

            item = {
                "hr": hr_full[t0:t0 + w],
                "lr": lr_full,
                "cond": cond_full[t0:t0 + w],
                "K": torch.as_tensor(f["K"][case_idx], dtype=torch.float32),
                "H": torch.as_tensor(f["H"][case_idx], dtype=torch.float32),
                "m": torch.as_tensor(f["m"][case_idx], dtype=torch.float32),
                "inj": torch.as_tensor(f["inj"][case_idx], dtype=torch.bool),
                "prod": torch.as_tensor(f["prod"][case_idx], dtype=torch.bool),
                "case_idx": torch.tensor(case_idx, dtype=torch.long),
                "t0": torch.tensor(t0, dtype=torch.long),
            }

            for key in [
                "Qc", "Qw", "watercut", "dt_frame", "dt_substep", "traj", "time",
                "valid_mask", "valid_lr_mask", "breakthrough_reached", "breakthrough_frame",
                "breakthrough_time", "n_valid_frames",
            ]:
                if key in self.available_keys:
                    arr = f[key][case_idx]
                    arr = self._slice_optional_time(arr, t0)
                    item[key] = self._to_tensor(key, arr)

        return item


train_ds = H5FiltrationPiRDDataset(TRAIN_H5, cfg)
val_ds = H5FiltrationPiRDDataset(VAL_H5, cfg)
test_ds = H5FiltrationPiRDDataset(TEST_H5, cfg)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("train/val/test windows:", len(train_ds), len(val_ds), len(test_ds))
print("train cases:", train_ds.length, "val cases:", val_ds.length, "test cases:", test_ds.length)

batch = next(iter(train_loader))
print("batch keys:", list(batch.keys()))
print("hr  :", tuple(batch["hr"].shape))
print("cond:", tuple(batch["cond"].shape))
print("lr  :", tuple(batch["lr"].shape))
print("K   :", tuple(batch["K"].shape))


## 6. Вспомогательные функции диффузии

In [ ]:

class ChannelNorm(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer("mean", mean.view(1, 1, 2, 1, 1))
        self.register_buffer("std", std.view(1, 1, 2, 1, 1))

    def encode(self, x):
        return (x - self.mean) / self.std

    def decode(self, x):
        return x * self.std + self.mean

def linear_beta_schedule(T, beta_start, beta_end, device):
    betas = torch.linspace(beta_start, beta_end, T, device=device)
    alphas = 1.0 - betas
    alphas_bar = torch.cumprod(alphas, dim=0)
    return betas, alphas, alphas_bar

def extract(a, t, x_shape):
    out = a.gather(0, t)
    return out.view(-1, *([1] * (len(x_shape) - 1)))

def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / max(half - 1, 1))
    ang = t.float()[:, None] * freqs[None, :]
    emb = torch.cat([torch.sin(ang), torch.cos(ang)], dim=1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb

betas, alphas, alphas_bar = linear_beta_schedule(cfg.diffusion_steps, cfg.beta_start, cfg.beta_end, device)
sqrt_alphas_bar = torch.sqrt(alphas_bar)
sqrt_one_minus_alphas_bar = torch.sqrt(1.0 - alphas_bar)


## 7. Conditional U-Net, максимально приближённый к `Diffusion/u_net_attention.py` из PiRD

In [ ]:
def get_timestep_embedding_pird(timesteps, embedding_dim):
    assert len(timesteps.shape) == 1
    half_dim = embedding_dim // 2
    scale = math.log(10000) / max(half_dim - 1, 1)
    emb = torch.exp(torch.arange(half_dim, dtype=torch.float32, device=timesteps.device) * -scale)
    emb = timesteps.float()[:, None] * emb[None, :]
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    if embedding_dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb

def pird_nonlinearity(x):
    return x * torch.sigmoid(x)

def pird_group_count(c):
    for g in [8, 4, 2, 1]:
        if c % g == 0:
            return g
    return 1

def NormalizePird(in_channels):
    return nn.GroupNorm(num_groups=pird_group_count(in_channels), num_channels=in_channels, eps=1e-6, affine=True)

class UpsamplePird(nn.Module):
    def __init__(self, in_channels, with_conv=True):
        super().__init__()
        self.with_conv = with_conv
        if with_conv:
            self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1, padding_mode="circular")

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2.0, mode="nearest")
        if self.with_conv:
            x = self.conv(x)
        return x

class DownsamplePird(nn.Module):
    def __init__(self, in_channels, with_conv=True):
        super().__init__()
        self.with_conv = with_conv
        if with_conv:
            self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=2, padding=0)

    def forward(self, x):
        if self.with_conv:
            x = F.pad(x, (0, 1, 0, 1), mode="circular")
            x = self.conv(x)
        else:
            x = F.avg_pool2d(x, kernel_size=2, stride=2)
        return x

class ResnetBlockPird(nn.Module):
    def __init__(self, in_channels, out_channels=None, dropout=0.0, temb_channels=512, conv_shortcut=False):
        super().__init__()
        out_channels = in_channels if out_channels is None else out_channels
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.use_conv_shortcut = conv_shortcut

        self.norm1 = NormalizePird(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, padding_mode="circular")
        self.temb_proj = nn.Linear(temb_channels, out_channels)
        self.norm2 = NormalizePird(out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, padding_mode="circular")

        if in_channels != out_channels:
            if conv_shortcut:
                self.conv_shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, padding_mode="circular")
            else:
                self.nin_shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, x, temb=None):
        h = self.conv1(pird_nonlinearity(self.norm1(x)))
        if temb is not None:
            h = h + self.temb_proj(pird_nonlinearity(temb))[:, :, None, None]
        h = self.conv2(self.dropout(pird_nonlinearity(self.norm2(h))))

        if self.in_channels != self.out_channels:
            if self.use_conv_shortcut:
                x = self.conv_shortcut(x)
            else:
                x = self.nin_shortcut(x)
        return x + h

class AttnBlockPird(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.norm = NormalizePird(in_channels)
        self.q = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.k = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.v = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.proj_out = nn.Conv2d(in_channels, in_channels, kernel_size=1)

    def forward(self, x):
        h_ = self.norm(x)
        q = self.q(h_)
        k = self.k(h_)
        v = self.v(h_)

        b, c, h, w = q.shape
        q = q.reshape(b, c, h * w).permute(0, 2, 1)
        k = k.reshape(b, c, h * w)
        attn = torch.bmm(q, k) * (c ** -0.5)
        attn = torch.softmax(attn, dim=2)

        v = v.reshape(b, c, h * w)
        attn = attn.permute(0, 2, 1)
        h_ = torch.bmm(v, attn).reshape(b, c, h, w)
        h_ = self.proj_out(h_)
        return x + h_

class PiRDAttentionUNet(nn.Module):
    def __init__(
        self,
        mean,
        std,
        ctx_frames,
        base_ch=48,
        ch_mult=(1, 2, 2),
        num_res_blocks=1,
        attn_resolutions=(16, 32),
        dropout=0.0,
        resamp_with_conv=True,
    ):
        super().__init__()
        self.norm = ChannelNorm(mean, std)
        self.ctx_frames = ctx_frames
        self.ch = base_ch
        self.temb_ch = base_ch * 4
        self.num_resolutions = len(ch_mult)
        self.num_res_blocks = num_res_blocks
        self.resolution = cfg.nx_hr
        self.in_channels = ctx_frames * 2
        self.out_channels = ctx_frames * 2

        self.temb = nn.Module()
        self.temb.dense = nn.ModuleList([
            nn.Linear(self.ch, self.temb_ch),
            nn.Linear(self.temb_ch, self.temb_ch),
        ])

        self.emb_conv = nn.Sequential(
            nn.Conv2d(self.in_channels, self.ch, kernel_size=1, stride=1, padding=0),
            nn.GELU(),
            nn.Conv2d(self.ch, self.ch, kernel_size=3, stride=1, padding=1, padding_mode="circular"),
        )

        self.conv_in = nn.Conv2d(self.in_channels, self.ch, kernel_size=3, stride=1, padding=1, padding_mode="circular")
        self.combine_conv = nn.Conv2d(self.ch * 2, self.ch, kernel_size=1, stride=1, padding=0)

        curr_res = self.resolution
        in_ch_mult = (1,) + tuple(ch_mult)
        self.down = nn.ModuleList()
        block_in = self.ch

        for i_level in range(self.num_resolutions):
            block = nn.ModuleList()
            attn = nn.ModuleList()
            block_in = self.ch * in_ch_mult[i_level]
            block_out = self.ch * ch_mult[i_level]
            for _ in range(self.num_res_blocks):
                block.append(
                    ResnetBlockPird(
                        in_channels=block_in,
                        out_channels=block_out,
                        temb_channels=self.temb_ch,
                        dropout=dropout,
                    )
                )
                block_in = block_out
                if curr_res in attn_resolutions:
                    attn.append(AttnBlockPird(block_in))
            down = nn.Module()
            down.block = block
            down.attn = attn
            if i_level != self.num_resolutions - 1:
                down.downsample = DownsamplePird(block_in, resamp_with_conv)
                curr_res = curr_res // 2
            self.down.append(down)

        self.mid = nn.Module()
        self.mid.block_1 = ResnetBlockPird(block_in, block_in, temb_channels=self.temb_ch, dropout=dropout)
        self.mid.attn_1 = AttnBlockPird(block_in)
        self.mid.block_2 = ResnetBlockPird(block_in, block_in, temb_channels=self.temb_ch, dropout=dropout)

        self.up = nn.ModuleList()
        for i_level in reversed(range(self.num_resolutions)):
            block = nn.ModuleList()
            attn = nn.ModuleList()
            block_out = self.ch * ch_mult[i_level]
            skip_in = self.ch * ch_mult[i_level]
            for i_block in range(self.num_res_blocks + 1):
                if i_block == self.num_res_blocks:
                    skip_in = self.ch * in_ch_mult[i_level]
                block.append(
                    ResnetBlockPird(
                        in_channels=block_in + skip_in,
                        out_channels=block_out,
                        temb_channels=self.temb_ch,
                        dropout=dropout,
                    )
                )
                block_in = block_out
                if curr_res in attn_resolutions:
                    attn.append(AttnBlockPird(block_in))
            up = nn.Module()
            up.block = block
            up.attn = attn
            if i_level != 0:
                up.upsample = UpsamplePird(block_in, resamp_with_conv)
                curr_res = curr_res * 2
            self.up.insert(0, up)

        self.norm_out = NormalizePird(block_in)
        self.conv_out = nn.Conv2d(block_in, self.out_channels, kernel_size=3, stride=1, padding=1, padding_mode="circular")

    def flatten_win(self, x):
        b, w, c, h, k = x.shape
        return x.reshape(b, w * c, h, k)

    def unflatten_win(self, x):
        b, c, h, w = x.shape
        return x.view(b, self.ctx_frames, 2, h, w)

    def forward(self, x_t, cond, t):
        assert x_t.shape[-1] == self.resolution and x_t.shape[-2] == self.resolution

        cond_n = self.norm.encode(cond)
        x_t_n = x_t / self.norm.std

        x_t_flat = self.flatten_win(x_t_n)
        cond_flat = self.flatten_win(cond_n)

        temb = get_timestep_embedding_pird(t, self.ch)
        temb = self.temb.dense[0](temb)
        temb = pird_nonlinearity(temb)
        temb = self.temb.dense[1](temb)

        x = self.conv_in(x_t_flat)
        cond_emb = self.emb_conv(cond_flat)
        x = self.combine_conv(torch.cat([x, cond_emb], dim=1))

        hs = [x]
        for i_level in range(self.num_resolutions):
            for i_block in range(self.num_res_blocks):
                h = self.down[i_level].block[i_block](hs[-1], temb)
                if len(self.down[i_level].attn) > 0:
                    h = self.down[i_level].attn[i_block](h)
                hs.append(h)
            if i_level != self.num_resolutions - 1:
                hs.append(self.down[i_level].downsample(hs[-1]))

        h = hs[-1]
        h = self.mid.block_1(h, temb)
        h = self.mid.attn_1(h)
        h = self.mid.block_2(h, temb)

        for i_level in reversed(range(self.num_resolutions)):
            for i_block in range(self.num_res_blocks + 1):
                h = self.up[i_level].block[i_block](torch.cat([h, hs.pop()], dim=1), temb)
                if len(self.up[i_level].attn) > 0:
                    h = self.up[i_level].attn[i_block](h)
            if i_level != 0:
                h = self.up[i_level].upsample(h)

        h = self.conv_out(pird_nonlinearity(self.norm_out(h)))
        h = self.unflatten_win(h)
        return h * self.norm.std

model = PiRDAttentionUNet(
    mean,
    std,
    ctx_frames=cfg.ctx_frames,
    base_ch=48,
    ch_mult=(1, 2, 2),
    num_res_blocks=1,
    attn_resolutions=(16, 32),
    dropout=0.0,
    resamp_with_conv=True,
).to(device)

print("params [M]:", sum(p.numel() for p in model.parameters()) / 1e6)

## 8. Diffusion objective и sampling

In [ ]:

def relperm_torch(S, cfg):
    Se = torch.clamp((S - cfg.Swc) / (cfg.Smax - cfg.Swc), 0.0, 1.0)
    Kw = torch.where(S <= cfg.Swc, torch.zeros_like(S), Se ** cfg.n_w)
    Kn = torch.where(S >= cfg.Smax, torch.zeros_like(S), (1.0 - Se) ** cfg.n_n)
    return Kw, Kn

def fw_from_s_torch(S, cfg):
    Kw, Kn = relperm_torch(S, cfg)
    lam_w = Kw / cfg.mu_w
    lam_n = Kn / cfg.mu_n
    lam_t = lam_w + lam_n + 1e-12
    return lam_w / lam_t, lam_t

def grad_xy(z):
    dx = z[..., :, 1:] - z[..., :, :-1]
    dy = z[..., 1:, :] - z[..., :-1, :]
    return dx, dy

def soft_front_mask(S, cfg):
    theta = cfg.Swc + cfg.front_theta * (cfg.Smax - cfg.Swc)
    return torch.sigmoid(cfg.front_k * (S - theta))

def physics_weight(epoch, cfg):
    if epoch < cfg.physics_start_epoch:
        return 0.0
    frac = (epoch - cfg.physics_start_epoch + 1) / max(1, cfg.physics_ramp_epochs)
    frac = min(1.0, max(0.0, frac))
    return cfg.max_phys_w * (frac ** 2)

def enforce_bc_batch(state, inj_mask, prod_mask, cfg):
    P = state[:, :, 0]
    S = state[:, :, 1]
    b, t, ny, nx = P.shape
    boundary2d = torch.zeros((b, ny, nx), dtype=torch.bool, device=state.device)
    boundary2d[:, 0, :] = True
    boundary2d[:, -1, :] = True
    boundary2d[:, :, 0] = True
    boundary2d[:, :, -1] = True
    boundary = boundary2d[:, None, :, :]
    inj = inj_mask[:, None, :, :]
    prod = prod_mask[:, None, :, :]
    P = torch.where(boundary, torch.full_like(P, cfg.Pgamma), P)
    P = torch.where(inj, torch.full_like(P, cfg.Pinj), P)
    P = torch.where(prod, torch.full_like(P, cfg.Pprod), P)
    S = torch.where(inj, torch.full_like(S, cfg.Smax), S)
    S = torch.clamp(S, cfg.Swc + 1e-4, cfg.Smax)
    return torch.stack([P, S], dim=2)

def total_flux_faces_torch(P, a, cfg):
    b, t, ny, nx = P.shape
    qx = torch.zeros(b, t, ny, nx + 1, device=P.device, dtype=P.dtype)
    qy = torch.zeros(b, t, ny + 1, nx, device=P.device, dtype=P.dtype)
    a_x = 0.5 * (a[:, :, :, :-1] + a[:, :, :, 1:])
    qx[:, :, :, 1:nx] = -a_x * (P[:, :, :, 1:] - P[:, :, :, :-1]) / cfg.dx
    qx[:, :, :, 0] = -a[:, :, :, 0] * (P[:, :, :, 0] - cfg.Pgamma) / (0.5 * cfg.dx)
    qx[:, :, :, nx] = -a[:, :, :, -1] * (cfg.Pgamma - P[:, :, :, -1]) / (0.5 * cfg.dx)
    a_y = 0.5 * (a[:, :, :-1, :] + a[:, :, 1:, :])
    qy[:, :, 1:ny, :] = -a_y * (P[:, :, 1:, :] - P[:, :, :-1, :]) / cfg.dy
    qy[:, :, 0, :] = -a[:, :, 0, :] * (P[:, :, 0, :] - cfg.Pgamma) / (0.5 * cfg.dy)
    qy[:, :, ny, :] = -a[:, :, -1, :] * (cfg.Pgamma - P[:, :, -1, :]) / (0.5 * cfg.dy)
    return qx, qy

def water_flux_faces_torch(S, qx, qy, cfg):
    b, t, ny, nx = S.shape
    fw, _ = fw_from_s_torch(S, cfg)
    fw_plus = float(fw_from_s_torch(torch.tensor([[[[cfg.S_plus]]]], device=S.device, dtype=S.dtype), cfg)[0].item())
    Fwx = torch.zeros(b, t, ny, nx + 1, device=S.device, dtype=S.dtype)
    Fwy = torch.zeros(b, t, ny + 1, nx, device=S.device, dtype=S.dtype)
    qx_int = qx[:, :, :, 1:nx]
    Fwx[:, :, :, 1:nx] = torch.where(qx_int >= 0, fw[:, :, :, :-1], fw[:, :, :, 1:]) * qx_int
    qy_int = qy[:, :, 1:ny, :]
    Fwy[:, :, 1:ny, :] = torch.where(qy_int >= 0, fw[:, :, :-1, :], fw[:, :, 1:, :]) * qy_int
    Fwx[:, :, :, 0] = torch.where(qx[:, :, :, 0] > 0, torch.full_like(qx[:, :, :, 0], fw_plus), fw[:, :, :, 0]) * qx[:, :, :, 0]
    Fwx[:, :, :, nx] = torch.where(qx[:, :, :, nx] < 0, torch.full_like(qx[:, :, :, nx], fw_plus), fw[:, :, :, -1]) * qx[:, :, :, nx]
    Fwy[:, :, 0, :] = torch.where(qy[:, :, 0, :] > 0, torch.full_like(qy[:, :, 0, :], fw_plus), fw[:, :, 0, :]) * qy[:, :, 0, :]
    Fwy[:, :, ny, :] = torch.where(qy[:, :, ny, :] < 0, torch.full_like(qy[:, :, ny, :], fw_plus), fw[:, :, -1, :]) * qy[:, :, ny, :]
    return Fwx, Fwy

def active_mask_torch(inj_mask, prod_mask):
    mask = torch.ones_like(inj_mask, dtype=torch.float32)
    mask[:, 0, :] = 0.0
    mask[:, -1, :] = 0.0
    mask[:, :, 0] = 0.0
    mask[:, :, -1] = 0.0
    mask = mask * (~inj_mask).float() * (~prod_mask).float()
    return mask

def masked_mean_sq(x, mask, eps=1e-12):
    while mask.dim() < x.dim():
        mask = mask.unsqueeze(1)
    return ((x ** 2) * mask).sum() / (mask.sum() * x.shape[1] + eps)

def physics_terms(pred, K_field, H_field, m_field, inj_mask, prod_mask, cfg):
    P = pred[:, :, 0]
    S = pred[:, :, 1]
    _, lam_t = fw_from_s_torch(S, cfg)
    a = K_field[:, None] * H_field[:, None] * lam_t
    qx, qy = total_flux_faces_torch(P, a, cfg)
    div_q = (qx[:, :, :, 1:] - qx[:, :, :, :-1]) / cfg.dx + (qy[:, :, 1:, :] - qy[:, :, :-1, :]) / cfg.dy
    Fwx, Fwy = water_flux_faces_torch(S, qx, qy, cfg)
    div_fw = (Fwx[:, :, :, 1:] - Fwx[:, :, :, :-1]) / cfg.dx + (Fwy[:, :, 1:, :] - Fwy[:, :, :-1, :]) / cfg.dy
    dSdt = (S[:, 1:] - S[:, :-1]) / cfg.dt
    res_s = m_field[:, None] * dSdt + div_fw[:, :-1]
    mask = active_mask_torch(inj_mask, prod_mask)
    phys_p = masked_mean_sq(div_q, mask)
    phys_s = masked_mean_sq(res_s, mask)
    return phys_p, phys_s

def full_field_error(pred, target, eps=1e-12):
    num = torch.sqrt(((pred - target) ** 2).mean())
    den = torch.sqrt((target ** 2).mean()) + eps
    return 100.0 * (num / den).item()

def metric_pack(pred, target):
    ff_total = full_field_error(pred, target)
    ff_p = full_field_error(pred[:, :, 0], target[:, :, 0])
    ff_s = full_field_error(pred[:, :, 1], target[:, :, 1])
    return {"ff_total": ff_total, "ff_P": ff_p, "ff_S": ff_s}

def target_score(metrics):
    return 0.3 * metrics["ff_P"] + 0.4 * metrics["ff_S"] + 0.3 * metrics["ff_total"]

def base_from_condition(cond, inj_mask, prod_mask, cfg):
    return enforce_bc_batch(cond, inj_mask, prod_mask, cfg)

def residual_target(hr, cond, inj_mask, prod_mask, cfg):
    hr_bc = enforce_bc_batch(hr, inj_mask, prod_mask, cfg)
    base = base_from_condition(cond, inj_mask, prod_mask, cfg)
    res = hr_bc - base
    res_p = torch.clamp(res[:, :, 0:1], -cfg.residual_clip_p, cfg.residual_clip_p)
    res_s = torch.clamp(res[:, :, 1:2], -cfg.residual_clip_s, cfg.residual_clip_s)
    return torch.cat([res_p, res_s], dim=2), hr_bc, base

def q_sample(x0, t, noise):
    return extract(sqrt_alphas_bar, t, x0.shape) * x0 + extract(sqrt_one_minus_alphas_bar, t, x0.shape) * noise

@torch.no_grad()
def p_sample_loop(model, cond, inj_mask, prod_mask, cfg):
    model.eval()
    b = cond.shape[0]
    x = torch.randn_like(cond)
    base = base_from_condition(cond, inj_mask, prod_mask, cfg)
    for i in reversed(range(cfg.diffusion_steps)):
        t = torch.full((b,), i, device=cond.device, dtype=torch.long)
        pred = model(x, cond, t)
        if cfg.predict_type == "eps":
            eps = pred
            a_bar = extract(alphas_bar, t, x.shape)
            sqrt_a_bar = extract(sqrt_alphas_bar, t, x.shape)
            sqrt_om = extract(sqrt_one_minus_alphas_bar, t, x.shape)
            x0 = (x - sqrt_om * eps) / sqrt_a_bar.clamp_min(1e-8)
            eps_used = eps
        else:
            x0 = pred
            a_bar = extract(alphas_bar, t, x.shape)
            eps_used = (x - torch.sqrt(a_bar) * x0) / torch.sqrt(1 - a_bar).clamp_min(1e-8)

        x0 = torch.cat([
            torch.clamp(x0[:, :, 0:1], -cfg.residual_clip_p, cfg.residual_clip_p),
            torch.clamp(x0[:, :, 1:2], -cfg.residual_clip_s, cfg.residual_clip_s),
        ], dim=2)

        if i > 0:
            a = extract(alphas, t, x.shape)
            a_bar = extract(alphas_bar, t, x.shape)
            beta = extract(betas, t, x.shape)
            noise = torch.randn_like(x)
            mean = (1 / torch.sqrt(a)) * (x - beta / torch.sqrt(1 - a_bar).clamp_min(1e-8) * eps_used)
            x = mean + torch.sqrt(beta) * noise
        else:
            x = x0

    pred = base + x
    pred = enforce_bc_batch(pred, inj_mask, prod_mask, cfg)
    return pred, base


## 9. Общая функция потерь PiRD

In [ ]:

def diffusion_reconstruction_loss(model, hr, cond, K_field, H_field, m_field, inj_mask, prod_mask, epoch, cfg):
    x0, hr_bc, base = residual_target(hr, cond, inj_mask, prod_mask, cfg)
    b = hr.shape[0]
    t = torch.randint(0, cfg.diffusion_steps, (b,), device=hr.device).long()
    noise = torch.randn_like(x0)
    x_t = q_sample(x0, t, noise)
    pred = model(x_t, cond, t)

    if cfg.predict_type == "eps":
        diff_loss = F.mse_loss(pred, noise)
        x0_hat = (x_t - extract(sqrt_one_minus_alphas_bar, t, x_t.shape) * pred) / extract(sqrt_alphas_bar, t, x_t.shape).clamp_min(1e-8)
    else:
        diff_loss = F.mse_loss(pred, x0)
        x0_hat = pred

    x0_hat = torch.cat([
        torch.clamp(x0_hat[:, :, 0:1], -cfg.residual_clip_p, cfg.residual_clip_p),
        torch.clamp(x0_hat[:, :, 1:2], -cfg.residual_clip_s, cfg.residual_clip_s),
    ], dim=2)

    pred_hr = enforce_bc_batch(base + x0_hat, inj_mask, prod_mask, cfg)
    rec = F.l1_loss(pred_hr, hr_bc)

    gp_pred_x, gp_pred_y = grad_xy(pred_hr[:, :, 0])
    gp_hr_x, gp_hr_y = grad_xy(hr_bc[:, :, 0])
    gs_pred_x, gs_pred_y = grad_xy(pred_hr[:, :, 1])
    gs_hr_x, gs_hr_y = grad_xy(hr_bc[:, :, 1])

    grad_p = F.l1_loss(gp_pred_x, gp_hr_x) + F.l1_loss(gp_pred_y, gp_hr_y)
    grad_s = F.l1_loss(gs_pred_x, gs_hr_x) + F.l1_loss(gs_pred_y, gs_hr_y)
    front_loss = F.l1_loss(soft_front_mask(pred_hr[:, :, 1], cfg), soft_front_mask(hr_bc[:, :, 1], cfg))
    residual_reg = (x0_hat ** 2).mean()
    identity_reg = F.l1_loss(pred_hr, base)

    phys_p, phys_s = physics_terms(pred_hr, K_field, H_field, m_field, inj_mask, prod_mask, cfg)
    phys = physics_weight(epoch, cfg) * phys_s

    loss = (
        cfg.loss_diffusion * diff_loss +
        cfg.loss_recon * rec +
        cfg.loss_p_grad * grad_p +
        cfg.loss_s_grad * grad_s +
        cfg.loss_s_front * front_loss +
        cfg.loss_residual_reg * residual_reg +
        cfg.loss_identity * identity_reg +
        phys
    )

    return loss, pred_hr, hr_bc, base, {
        "diff": float(diff_loss.item()),
        "rec": float(rec.item()),
        "grad_p": float(grad_p.item()),
        "grad_s": float(grad_s.item()),
        "front_s": float(front_loss.item()),
        "residual_reg": float(residual_reg.item()),
        "identity": float(identity_reg.item()),
        "phys": float(phys.item()),
        "phys_p": float(phys_p.item()),
        "phys_s": float(phys_s.item()),
        "phys_w": physics_weight(epoch, cfg),
    }


## 10. Обучение и валидация

In [ ]:

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=cfg.lr_scheduler_factor,
    patience=cfg.lr_scheduler_patience, min_lr=cfg.lr_scheduler_min
)

@torch.no_grad()
def evaluate(model, loader, epoch, cfg):
    model.eval()
    rows = []
    for batch in loader:
        hr = batch["hr"].to(device)
        cond = batch["cond"].to(device)
        K_t = batch["K"].to(device)
        H_t = batch["H"].to(device)
        m_t = batch["m"].to(device)
        inj_t = batch["inj"].to(device)
        prod_t = batch["prod"].to(device)

        loss, pred, hr_bc, base, terms = diffusion_reconstruction_loss(
            model, hr, cond, K_t, H_t, m_t, inj_t, prod_t, epoch, cfg
        )
        pred_m = metric_pack(pred, hr_bc)
        base_m = metric_pack(base, hr_bc)

        rows.append({
            "loss": float(loss.item()),
            "ff_total": pred_m["ff_total"],
            "ff_P": pred_m["ff_P"],
            "ff_S": pred_m["ff_S"],
            "score": target_score(pred_m),
            "base_total": base_m["ff_total"],
            "base_P": base_m["ff_P"],
            "base_S": base_m["ff_S"],
            "phys": terms["phys"],
            "phys_p": terms["phys_p"],
            "phys_s": terms["phys_s"],
            "phys_w": terms["phys_w"],
            "front_s": terms["front_s"],
            "diff": terms["diff"],
        })

    keys = rows[0].keys()
    return {k: float(np.mean([r[k] for r in rows])) for k in keys}

def train(model, train_loader, val_loader, cfg):
    hist = {
        "train_loss": [], "val_loss": [],
        "val_ff_total": [], "val_ff_P": [], "val_ff_S": [],
        "val_score": [], "base_ff_total": [], "base_ff_P": [], "base_ff_S": [],
        "lr": [], "phys_w": [], "phys_p": [], "phys_s": [], "front_s": [], "diff": []
    }
    best = {"score": float("inf"), "state": None, "epoch": 0, "stats": None}
    bad_epochs = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        train_losses = []
        for batch in train_loader:
            hr = batch["hr"].to(device)
            cond = batch["cond"].to(device)
            K_t = batch["K"].to(device)
            H_t = batch["H"].to(device)
            m_t = batch["m"].to(device)
            inj_t = batch["inj"].to(device)
            prod_t = batch["prod"].to(device)

            optimizer.zero_grad(set_to_none=True)
            loss, pred, hr_bc, base, terms = diffusion_reconstruction_loss(
                model, hr, cond, K_t, H_t, m_t, inj_t, prod_t, epoch, cfg
            )
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(float(loss.item()))

        val_stats = evaluate(model, val_loader, epoch, cfg)
        scheduler.step(val_stats["score"])

        hist["train_loss"].append(float(np.mean(train_losses)))
        hist["val_loss"].append(val_stats["loss"])
        hist["val_ff_total"].append(val_stats["ff_total"])
        hist["val_ff_P"].append(val_stats["ff_P"])
        hist["val_ff_S"].append(val_stats["ff_S"])
        hist["val_score"].append(val_stats["score"])
        hist["base_ff_total"].append(val_stats["base_total"])
        hist["base_ff_P"].append(val_stats["base_P"])
        hist["base_ff_S"].append(val_stats["base_S"])
        hist["lr"].append(float(optimizer.param_groups[0]["lr"]))
        hist["phys_w"].append(val_stats["phys_w"])
        hist["phys_p"].append(val_stats["phys_p"])
        hist["phys_s"].append(val_stats["phys_s"])
        hist["front_s"].append(val_stats["front_s"])
        hist["diff"].append(val_stats["diff"])

        improved = val_stats["score"] < (best["score"] - cfg.early_stopping_min_delta)
        if improved:
            best = {
                "score": val_stats["score"],
                "state": copy.deepcopy(model.state_dict()),
                "epoch": epoch,
                "stats": val_stats,
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        print(
            f"[{epoch:03d}] train={hist['train_loss'][-1]:.5f} "
            f"val={val_stats['loss']:.5f} score={val_stats['score']:.3f} "
            f"ff(P,S,T)=({val_stats['ff_P']:.2f},{val_stats['ff_S']:.2f},{val_stats['ff_total']:.2f}) "
            f"base=({val_stats['base_P']:.2f},{val_stats['base_S']:.2f},{val_stats['base_total']:.2f}) "
            f"phys_w={val_stats['phys_w']:.2e}"
        )

        if bad_epochs >= cfg.early_stopping_patience:
            print("Early stopping.")
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"])
    return hist, best


## 11. Запуск обучения

In [ ]:
start = time.time()
history, best = train(model, train_loader, val_loader, cfg)
elapsed = time.time() - start

training_time_total_sec = float(elapsed)
training_epochs_completed = len(history.get("train_loss", []))
training_time_per_epoch_sec = training_time_total_sec / max(1, training_epochs_completed)

print("best epoch:", best["epoch"])
print("best stats:", best["stats"])
print("elapsed [s]:", round(training_time_total_sec, 2))
print("sec/epoch:", round(training_time_per_epoch_sec, 2))


In [ ]:

epochs = np.arange(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(10, 4))
plt.plot(epochs, history["train_loss"], label="train loss")
plt.plot(epochs, history["val_loss"], label="val loss")
plt.legend(); plt.grid(True); plt.title("Loss"); plt.show()

plt.figure(figsize=(10, 4))
plt.plot(epochs, history["val_ff_P"], label="PiRD P")
plt.plot(epochs, history["val_ff_S"], label="PiRD S")
plt.plot(epochs, history["val_ff_total"], label="PiRD total")
plt.plot(epochs, history["base_ff_total"], "--", label="baseline total")
plt.legend(); plt.grid(True); plt.title("Full-field error, %"); plt.show()


## 12. Тест: сравнение baseline / PiRD / HR

In [ ]:

@torch.no_grad()
def run_test_example(model, test_loader, cfg, sample_idx=0):
    model.eval()
    for i, batch in enumerate(test_loader):
        if i != sample_idx:
            continue
        hr = batch["hr"].to(device)
        cond = batch["cond"].to(device)
        inj_t = batch["inj"].to(device)
        prod_t = batch["prod"].to(device)
        pred, base = p_sample_loop(model, cond, inj_t, prod_t, cfg)
        hr_bc = enforce_bc_batch(hr, inj_t, prod_t, cfg)
        return {
            "pred": pred.cpu(),
            "base": base.cpu(),
            "hr": hr_bc.cpu(),
            "metrics_pred": metric_pack(pred, hr_bc),
            "metrics_base": metric_pack(base, hr_bc),
        }

result = run_test_example(model, test_loader, cfg, sample_idx=0)
print("baseline:", result["metrics_base"])
print("pird    :", result["metrics_pred"])


In [ ]:

pred = result["pred"][0]
base = result["base"][0]
hr = result["hr"][0]
mid_t = cfg.ctx_frames - 1

fig, ax = plt.subplots(2, 4, figsize=(15, 7))
ax[0,0].imshow(base[mid_t,0], cmap="viridis"); ax[0,0].set_title("Baseline P"); ax[0,0].axis("off")
ax[0,1].imshow(pred[mid_t,0], cmap="viridis"); ax[0,1].set_title("PiRD P"); ax[0,1].axis("off")
ax[0,2].imshow(hr[mid_t,0], cmap="viridis"); ax[0,2].set_title("HR P"); ax[0,2].axis("off")
ax[0,3].imshow((pred[mid_t,0]-hr[mid_t,0]).abs(), cmap="magma"); ax[0,3].set_title("|Err| P"); ax[0,3].axis("off")

ax[1,0].imshow(base[mid_t,1], cmap="turbo"); ax[1,0].set_title("Baseline S"); ax[1,0].axis("off")
ax[1,1].imshow(pred[mid_t,1], cmap="turbo"); ax[1,1].set_title("PiRD S"); ax[1,1].axis("off")
ax[1,2].imshow(hr[mid_t,1], cmap="turbo"); ax[1,2].set_title("HR S"); ax[1,2].axis("off")
ax[1,3].imshow((pred[mid_t,1]-hr[mid_t,1]).abs(), cmap="magma"); ax[1,3].set_title("|Err| S"); ax[1,3].axis("off")
plt.tight_layout()


## 13. Сводка по порогам качества

In [ ]:

def threshold_report(metrics, cfg):
    return {
        "P<5%": metrics["ff_P"] < cfg.target_ff_P,
        "S<8%": metrics["ff_S"] < cfg.target_ff_S,
        "joint<7%": metrics["ff_total"] < cfg.target_ff_joint,
    }

summary_rows = [
    {"model": "baseline", **result["metrics_base"], **threshold_report(result["metrics_base"], cfg)},
    {"model": "pird", **result["metrics_pred"], **threshold_report(result["metrics_pred"], cfg)},
]
summary_rows


## 14. Comprehensive metrics and JSON export

Блок ниже собирает метрики по тестовому HDF5-датасету: точность PiRD и baseline, ошибки в сложных зонах, proxy по фронту/массе, скорость инференса, память и краткую информацию об обучении.


In [ ]:
METRICS_DIR = Path("metrics")
METRICS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_JSON = METRICS_DIR / f"{NAME}_pird_metrics.json"


def _sync_cuda():
    if str(device).startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()


def _to_float(x):
    if isinstance(x, torch.Tensor):
        return float(x.detach().cpu().item())
    if isinstance(x, (np.floating, np.integer)):
        return float(x)
    return float(x)


def model_num_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": int(total), "trainable": int(trainable)}


@torch.no_grad()
def rel_l2_percent(pred, target, eps=1e-12):
    return _to_float(100.0 * torch.linalg.vector_norm(pred - target) / (torch.linalg.vector_norm(target) + eps))


@torch.no_grad()
def rmse_value(pred, target):
    return _to_float(torch.sqrt(torch.mean((pred - target) ** 2)))


@torch.no_grad()
def mae_value(pred, target):
    return _to_float(torch.mean(torch.abs(pred - target)))


@torch.no_grad()
def normalize_field(x, mean, std):
    mean_t = mean.to(x.device).view(1, 1, -1, 1, 1)
    std_t = std.to(x.device).view(1, 1, -1, 1, 1)
    return (x - mean_t) / (std_t + 1e-8)


@torch.no_grad()
def full_field_metrics(pred, target, prefix="", mean=None, std=None):
    metrics = {
        f"{prefix}rel_l2_full_raw_pct": rel_l2_percent(pred, target),
        f"{prefix}rel_l2_P_pct": rel_l2_percent(pred[:, :, 0], target[:, :, 0]),
        f"{prefix}rel_l2_S_pct": rel_l2_percent(pred[:, :, 1], target[:, :, 1]),
        f"{prefix}rmse_full_raw": rmse_value(pred, target),
        f"{prefix}rmse_P": rmse_value(pred[:, :, 0], target[:, :, 0]),
        f"{prefix}rmse_S": rmse_value(pred[:, :, 1], target[:, :, 1]),
        f"{prefix}mae_full_raw": mae_value(pred, target),
        f"{prefix}mae_P": mae_value(pred[:, :, 0], target[:, :, 0]),
        f"{prefix}mae_S": mae_value(pred[:, :, 1], target[:, :, 1]),
    }

    p_rel = metrics[f"{prefix}rel_l2_P_pct"]
    s_rel = metrics[f"{prefix}rel_l2_S_pct"]
    metrics[f"{prefix}rel_l2_full_balanced_pct"] = 0.5 * (p_rel + s_rel)

    if mean is not None and std is not None:
        pred_n = normalize_field(pred, mean, std)
        target_n = normalize_field(target, mean, std)
        metrics[f"{prefix}rel_l2_full_norm_pct"] = rel_l2_percent(pred_n, target_n)
        metrics[f"{prefix}rmse_full_norm"] = rmse_value(pred_n, target_n)

    return metrics


@torch.no_grad()
def masked_metric_value(fn, pred, target, mask):
    mask = mask.float()
    while mask.dim() < pred.dim():
        mask = mask.unsqueeze(2)
    mask = mask.expand_as(pred)
    denom = mask.sum().clamp_min(1.0)

    if fn == "rmse":
        return _to_float(torch.sqrt((((pred - target) ** 2) * mask).sum() / denom))
    if fn == "mae":
        return _to_float((torch.abs(pred - target) * mask).sum() / denom)

    num = torch.sqrt((((pred - target) ** 2) * mask).sum() / denom)
    den = torch.sqrt(((target ** 2) * mask).sum() / denom) + 1e-12
    return _to_float(100.0 * num / den)


@torch.no_grad()
def make_complex_zone_mask(hr, K, cfg):
    S = hr[:, :, 1]
    front = (S > (cfg.Swc + 0.04)) & (S < (cfg.Smax - 0.04))

    gx = torch.zeros_like(S)
    gy = torch.zeros_like(S)
    gx[..., :-1] = S[..., 1:] - S[..., :-1]
    gy[..., :-1, :] = S[..., 1:, :] - S[..., :-1, :]
    gmag = torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)
    b, t, h, w = gmag.shape
    gthr = torch.quantile(gmag.reshape(b, t, -1), 0.75, dim=-1, keepdim=True).view(b, t, 1, 1)
    grad_zone = gmag >= gthr

    logK = torch.log(K.clamp_min(1e-8))
    kgx = torch.zeros_like(logK)
    kgy = torch.zeros_like(logK)
    kgx[..., :-1] = logK[..., 1:] - logK[..., :-1]
    kgy[..., :-1, :] = logK[..., 1:, :] - logK[..., :-1, :]
    kgmag = torch.sqrt(kgx ** 2 + kgy ** 2 + 1e-12)
    kth = torch.quantile(kgmag.reshape(b, -1), 0.80, dim=-1, keepdim=True).view(b, 1, 1)
    geo_zone = (kgmag >= kth).unsqueeze(1).expand(-1, t, -1, -1)
    return (front | grad_zone | geo_zone).float()


@torch.no_grad()
def front_iou(pred_s, target_s, cfg):
    theta = cfg.Swc + cfg.front_theta * (cfg.Smax - cfg.Swc)
    pred_front = pred_s >= theta
    target_front = target_s >= theta
    inter = (pred_front & target_front).float().sum()
    union = (pred_front | target_front).float().sum().clamp_min(1.0)
    return _to_float(inter / union)


@torch.no_grad()
def saturation_bounds_metrics(pred_s, cfg):
    low = (cfg.Swc - pred_s).clamp_min(0.0)
    high = (pred_s - cfg.Smax).clamp_min(0.0)
    violation = low + high
    return {
        "S_bounds_mean_violation": _to_float(violation.mean()),
        "S_bounds_max_violation": _to_float(violation.max()),
        "S_bounds_fraction_violated": _to_float((violation > 0).float().mean()),
    }


@torch.no_grad()
def mass_proxy_metrics(pred_s, base_s, target_s):
    pred_mass = pred_s.mean(dim=(-2, -1))
    base_mass = base_s.mean(dim=(-2, -1))
    target_mass = target_s.mean(dim=(-2, -1))
    return {
        "mass_proxy_accumulation_abs_error": mae_value(pred_mass, target_mass),
        "base_mass_proxy_accumulation_abs_error": mae_value(base_mass, target_mass),
    }


@torch.no_grad()
def forward_model_for_metrics(model, batch):
    hr = batch["hr"].to(device)
    cond = batch["cond"].to(device)
    inj = batch["inj"].to(device)
    prod = batch["prod"].to(device)
    pred, base = p_sample_loop(model, cond, inj, prod, cfg)
    hr_bc = enforce_bc_batch(hr, inj, prod, cfg)
    return pred, base, hr_bc


@torch.no_grad()
def forward_base_for_metrics(batch):
    cond = batch["cond"].to(device)
    inj = batch["inj"].to(device)
    prod = batch["prod"].to(device)
    return base_from_condition(cond, inj, prod, cfg)


@torch.no_grad()
def evaluate_full_metrics(model, loader, cfg):
    model.eval()
    rows = []

    for batch_id, batch in enumerate(loader):
        pred, base, hr = forward_model_for_metrics(model, batch)
        K = batch["K"].to(device)
        complex_mask = make_complex_zone_mask(hr, K, cfg)

        row = {"batch_id": float(batch_id)}
        row.update(full_field_metrics(pred, hr, mean=mean, std=std))
        row.update(full_field_metrics(base, hr, prefix="base_", mean=mean, std=std))

        row["complex_rel_l2_full_raw_pct"] = masked_metric_value("rel_l2", pred, hr, complex_mask)
        row["complex_rmse_full_raw"] = masked_metric_value("rmse", pred, hr, complex_mask)
        row["base_complex_rel_l2_full_raw_pct"] = masked_metric_value("rel_l2", base, hr, complex_mask)
        row["base_complex_rmse_full_raw"] = masked_metric_value("rmse", base, hr, complex_mask)

        row["front_iou"] = front_iou(pred[:, :, 1], hr[:, :, 1], cfg)
        row["base_front_iou"] = front_iou(base[:, :, 1], hr[:, :, 1], cfg)
        row.update(saturation_bounds_metrics(pred[:, :, 1], cfg))
        row.update(mass_proxy_metrics(pred[:, :, 1], base[:, :, 1], hr[:, :, 1]))

        row["gain_rel_l2_full_raw"] = row["base_rel_l2_full_raw_pct"] - row["rel_l2_full_raw_pct"]
        row["gain_rel_l2_P"] = row["base_rel_l2_P_pct"] - row["rel_l2_P_pct"]
        row["gain_rel_l2_S"] = row["base_rel_l2_S_pct"] - row["rel_l2_S_pct"]
        row["gain_front_iou"] = row["front_iou"] - row["base_front_iou"]

        rows.append({k: float(v) for k, v in row.items()})

    summary = {}
    keys = sorted(set().union(*[r.keys() for r in rows])) if rows else []
    for key in keys:
        vals = [r[key] for r in rows if key in r and np.isfinite(r[key])]
        if vals:
            summary[key] = {
                "mean": float(np.mean(vals)),
                "std": float(np.std(vals)),
                "min": float(np.min(vals)),
                "max": float(np.max(vals)),
            }

    return summary, rows


def measure_inference_performance(model, loader, cfg, n_warmup=2, n_runs=20):
    model.eval()
    model_times, base_times = [], []

    if str(device).startswith("cuda") and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    process = psutil.Process() if psutil is not None else None
    rss_before = process.memory_info().rss if process is not None else None
    tracemalloc.start()

    with torch.no_grad():
        it = iter(loader)
        for _ in range(n_warmup):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(loader)
                batch = next(it)
            _ = forward_model_for_metrics(model, batch)
            _ = forward_base_for_metrics(batch)

        _sync_cuda()
        it = iter(loader)
        for _ in range(n_runs):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(loader)
                batch = next(it)

            _sync_cuda()
            t0 = time.perf_counter()
            _ = forward_model_for_metrics(model, batch)
            _sync_cuda()
            model_times.append(time.perf_counter() - t0)

            _sync_cuda()
            t0 = time.perf_counter()
            _ = forward_base_for_metrics(batch)
            _sync_cuda()
            base_times.append(time.perf_counter() - t0)

    current, peak_py = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    rss_after = process.memory_info().rss if process is not None else None

    gpu = {}
    if str(device).startswith("cuda") and torch.cuda.is_available():
        gpu = {
            "peak_allocated_mb": float(torch.cuda.max_memory_allocated() / 1024**2),
            "peak_reserved_mb": float(torch.cuda.max_memory_reserved() / 1024**2),
        }

    batch_size = int(getattr(cfg, "batch_size", 1))
    model_mean = float(np.mean(model_times))
    base_mean = float(np.mean(base_times))

    return {
        "batch_size": batch_size,
        "inference_time_sec_per_batch_mean": model_mean,
        "inference_time_sec_per_batch_std": float(np.std(model_times)),
        "inference_time_sec_per_batch_p50": float(np.percentile(model_times, 50)),
        "inference_time_sec_per_batch_p95": float(np.percentile(model_times, 95)),
        "inference_time_sec_per_sample_mean": model_mean / max(1, batch_size),
        "throughput_samples_per_sec": float(max(1, batch_size) / max(model_mean, 1e-12)),
        "base_interpolation_time_sec_per_batch_mean": base_mean,
        "base_interpolation_time_sec_per_batch_std": float(np.std(base_times)),
        "base_interpolation_time_sec_per_sample_mean": base_mean / max(1, batch_size),
        "base_interpolation_speedup_vs_model": float(model_mean / max(base_mean, 1e-12)),
        "python_peak_tracemalloc_mb": float(peak_py / 1024**2),
        "cpu_rss_delta_mb": None if rss_before is None else float((rss_after - rss_before) / 1024**2),
        "gpu": gpu,
    }


if "best" in globals() and isinstance(best, dict) and best.get("state") is not None:
    model.load_state_dict(best["state"])

accuracy_summary, accuracy_rows = evaluate_full_metrics(model, test_loader, cfg)
performance_summary = measure_inference_performance(
    model,
    test_loader,
    cfg,
    n_warmup=2,
    n_runs=min(20, max(1, len(test_loader))),
)

training_summary = {
    "training_time_total_sec": globals().get("training_time_total_sec"),
    "training_time_per_epoch_sec": globals().get("training_time_per_epoch_sec"),
    "epochs_completed": int(globals().get("training_epochs_completed", len(history.get("train_loss", [])) if "history" in globals() else 0)),
    "best_epoch": int(best.get("epoch", 0)) if "best" in globals() and isinstance(best, dict) else None,
    "best_score": float(best.get("score", np.nan)) if "best" in globals() and isinstance(best, dict) else None,
    "history": history if "history" in globals() else None,
}

cfg_dict = dict(getattr(cfg, "__dict__", {}))
for k, v in list(cfg_dict.items()):
    if isinstance(v, (np.integer, np.floating)):
        cfg_dict[k] = float(v)
    elif isinstance(v, torch.device):
        cfg_dict[k] = str(v)

results = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset": {
        "name": NAME,
        "data_dir": str(DATA_DIR),
        "train_h5": str(TRAIN_H5),
        "val_h5": str(VAL_H5),
        "test_h5": str(TEST_H5),
        "train_cases": train_ds.length,
        "val_cases": val_ds.length,
        "test_cases": test_ds.length,
        "train_windows": len(train_ds),
        "val_windows": len(val_ds),
        "test_windows": len(test_ds),
    },
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "device": str(device),
    },
    "config": cfg_dict,
    "model": model_num_parameters(model),
    "training": training_summary,
    "performance": performance_summary,
    "accuracy_summary": accuracy_summary,
    "accuracy_per_batch": accuracy_rows,
}

with open(METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved metrics to: {METRICS_JSON}")
print("\n=== Key metrics ===")
print("Params:", results["model"])
print("Inference sec/sample:", performance_summary["inference_time_sec_per_sample_mean"])
print("Base interpolation sec/sample:", performance_summary["base_interpolation_time_sec_per_sample_mean"])
print("Peak GPU allocated MB:", performance_summary["gpu"].get("peak_allocated_mb") if performance_summary["gpu"] else None)
print("Training total sec:", training_summary["training_time_total_sec"])

for key in [
    "rel_l2_full_raw_pct",
    "rel_l2_full_balanced_pct",
    "rel_l2_full_norm_pct",
    "rel_l2_P_pct",
    "rel_l2_S_pct",
    "rmse_full_raw",
    "rmse_full_norm",
    "rmse_P",
    "rmse_S",
    "mae_P",
    "mae_S",
    "front_iou",
    "gain_rel_l2_full_raw",
    "gain_rel_l2_P",
    "gain_rel_l2_S",
    "complex_rel_l2_full_raw_pct",
    "S_bounds_mean_violation",
    "S_bounds_fraction_violated",
    "mass_proxy_accumulation_abs_error",
]:
    if key in accuracy_summary:
        print(f"{key}: {accuracy_summary[key]['mean']:.6g} ± {accuracy_summary[key]['std']:.6g}")


## 15. Что изменено в этой версии

1. Синтетическая генерация cases внутри PiRD-датасета заменена чтением `config.json`, `{NAME}_train.h5`, `{NAME}_val.h5`, `{NAME}_test.h5`.
2. `cfg` синхронизируется с HDF5-формами: `nt_hr`, `nx_hr`, `ny_hr`, `nt_lr`, `nx_lr`, `ny_lr`, `s_up`, `t_up`.
3. Условие `cond` строится из сохранённого LR-поля датасета через spatial+temporal interpolation до HR-размера.
4. DataLoader теперь возвращает окна `[batch, ctx_frames, 2, ny_hr, nx_hr]`, а также исходный `lr` и все доступные auxiliary-поля из HDF5.
5. Добавлен блок comprehensive metrics с экспортом в `metrics/{NAME}_pird_metrics.json`.
